# ASTRA — Milestone 1: DataCo Data Ingestion & Schema Validation

**Scope of this notebook:** data understanding and validation only.
No model training, no feature engineering for training, no full
graphical EDA (that's Milestone 2), no risk-category assignment.

Every output in this notebook comes from actually running the cells
against the real `DataCoSupplyChainDataset.csv` — nothing here is
typed-in or assumed.

## 1. Imports

In [1]:
from pathlib import Path
import pandas as pd

from src.data.loaders import load_csv, file_discovery_report
from src.data.validation import validate_dataco_schema
from src.utils.config import load_config, resolve_path
from src.utils.logger import get_logger

logger = get_logger(__name__)
pd.set_option("display.max_columns", 60)

## 2. Load configuration

Paths come from `config/settings.yaml`, not hardcoded, so this
notebook works the same regardless of where the project is checked
out.

In [2]:
config = load_config()
raw_data_dir = resolve_path(config["paths"]["data_raw"])
raw_data_dir

PosixPath('/home/claude/work/astra/data/raw')

### File discovery (Step 1)

Confirms the two approved files exist and are readable before
attempting to load them. `tokenized_access_logs.csv` is deliberately
NOT referenced anywhere in this notebook — it remains out of scope.

In [3]:
for fname in ["DataCoSupplyChainDataset.csv", "DescriptionDataCoSupplyChain.csv"]:
    report = file_discovery_report(raw_data_dir / fname)
    print(report)

{'filename': 'DataCoSupplyChainDataset.csv', 'path': '/home/claude/work/astra/data/raw/DataCoSupplyChainDataset.csv', 'exists': True, 'is_file': True, 'is_readable': True, 'size_bytes': 95910149, 'size_mb': 91.47}
{'filename': 'DescriptionDataCoSupplyChain.csv', 'path': '/home/claude/work/astra/data/raw/DescriptionDataCoSupplyChain.csv', 'exists': True, 'is_file': True, 'is_readable': True, 'size_bytes': 3444, 'size_mb': 0.0}


## 3. Load DataCo

Uses `src/data/loaders.load_csv`, which validates the file, tries
UTF-8 first, and falls back to Latin-1 if needed (the real file is not
valid UTF-8, as the warning below shows).

In [4]:
df = load_csv(raw_data_dir / "DataCoSupplyChainDataset.csv")
df.head(3)

2026-09-24 20:19:30 | WARNING  | src.data.loaders | Failed to decode DataCoSupplyChainDataset.csv with encoding=utf-8, trying next fallback.


2026-09-24 20:19:32 | INFO     | src.data.loaders | Loaded DataCoSupplyChainDataset.csv: 180519 rows x 53 columns (encoding=latin-1)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


## 4. Dataset dimensions

In [5]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory (deep): {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

Rows: 180,519
Columns: 53


Memory (deep): 298.92 MB


## 5. Column inspection

In [6]:
for col in df.columns:
    print(col)

Type
Days for shipping (real)
Days for shipment (scheduled)
Benefit per order
Sales per customer
Delivery Status
Late_delivery_risk
Category Id
Category Name
Customer City
Customer Country
Customer Email
Customer Fname
Customer Id
Customer Lname
Customer Password
Customer Segment
Customer State
Customer Street
Customer Zipcode
Department Id
Department Name
Latitude
Longitude
Market
Order City
Order Country
Order Customer Id
order date (DateOrders)
Order Id
Order Item Cardprod Id
Order Item Discount
Order Item Discount Rate
Order Item Id
Order Item Product Price
Order Item Profit Ratio
Order Item Quantity
Sales
Order Item Total
Order Profit Per Order
Order Region
Order State
Order Status
Order Zipcode
Product Card Id
Product Category Id
Product Description
Product Image
Product Name
Product Price
Product Status
shipping date (DateOrders)
Shipping Mode


## 6. Data types

In [7]:
df.dtypes

Type                              object
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                   object
Late_delivery_risk                 int64
Category Id                        int64
Category Name                     object
Customer City                     object
Customer Country                  object
Customer Email                    object
Customer Fname                    object
Customer Id                        int64
Customer Lname                    object
Customer Password                 object
Customer Segment                  object
Customer State                    object
Customer Street                   object
Customer Zipcode                 float64
Department Id                      int64
Department Name                   object
Latitude                         float64
Longitude                        float64
Market          

## 7. Missing values

In [8]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(3)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

,missing_count,missing_pct
Product Description,180519,100.000
Order Zipcode,155679,86.240
Customer Lname,8,0.004
Customer Zipcode,3,0.002


## 8. Duplicate analysis

In [9]:
print("Fully duplicate rows:", df.duplicated().sum())

# columns that are exact duplicates of another column under a different name
dup_pairs = [
    ("Order Customer Id", "Customer Id"),
    ("Product Category Id", "Category Id"),
    ("Order Item Cardprod Id", "Product Card Id"),
]
for a, b in dup_pairs:
    print(f"{a} == {b} for all rows? {(df[a] == df[b]).all()}")

Fully duplicate rows: 0
Order Customer Id == Customer Id for all rows? True
Product Category Id == Category Id for all rows? True
Order Item Cardprod Id == Product Card Id for all rows? True


## 9. Data dictionary comparison (Step 4)

Comparing the supplied `DescriptionDataCoSupplyChain.csv` against the
real column names — case- and whitespace-insensitively, since the
description file has trailing spaces and one case mismatch.

In [10]:
desc = load_csv(raw_data_dir / "DescriptionDataCoSupplyChain.csv")

dataset_cols = list(df.columns)
desc_fields_stripped = [f.strip() for f in desc["FIELDS"]]

dataset_lower = {c.strip().lower(): c for c in dataset_cols}
desc_lower = {f.lower(): f for f in desc_fields_stripped}

in_dataset_not_desc = [c for c in dataset_cols if c.strip().lower() not in desc_lower]
in_desc_not_dataset = [f for f in desc_fields_stripped if f.lower() not in dataset_lower]
case_mismatches = [
    (c, desc_lower[c.strip().lower()])
    for c in dataset_cols
    if c.strip().lower() in desc_lower and desc_lower[c.strip().lower()] != c
]

print("In dataset, not in description file:", in_dataset_not_desc)
print("In description file, not in dataset:", in_desc_not_dataset)
print("Case mismatches:", case_mismatches)

2026-09-24 20:19:33 | INFO     | src.data.loaders | Loaded DescriptionDataCoSupplyChain.csv: 52 rows x 2 columns (encoding=utf-8)


In dataset, not in description file: ['Order Zipcode']
In description file, not in dataset: []
Case mismatches: [('shipping date (DateOrders)', 'Shipping date (DateOrders)')]


The full per-column data dictionary (description, dtype, missingness, provisional role, notes) is generated as `artifacts/reports/dataco_data_dictionary.csv` — see that file for the complete table; it's too wide to display cleanly inline here.

In [11]:
data_dict = pd.read_csv(resolve_path(config["paths"]["artifacts_reports"]) / "dataco_data_dictionary.csv")
data_dict[["column_name", "pandas_dtype", "provisional_role", "missing_percentage", "unique_count"]]

,column_name,pandas_dtype,provisional_role,missing_percentage,unique_count
0,Type,object,CATEGORICAL,0.000,4
1,Days for shipping (real),int64,NUMERIC,0.000,7
2,Days for shipment (scheduled),int64,NUMERIC,0.000,4
3,Benefit per order,float64,NUMERIC,0.000,21998
4,Sales per customer,float64,NUMERIC,0.000,2927
5,Delivery Status,object,POSSIBLE_TARGET,0.000,4
6,Late_delivery_risk,int64,POSSIBLE_TARGET,0.000,2
7,Category Id,int64,IDENTIFIER,0.000,51
8,Category Name,object,CATEGORICAL,0.000,50
9,Customer City,object,CATEGORICAL,0.000,563


## 10. Date/time candidates (Step 5)

In [12]:
for col in ["order date (DateOrders)", "shipping date (DateOrders)"]:
    parsed = pd.to_datetime(df[col], errors="coerce")
    failed = int(parsed.isna().sum() - df[col].isna().sum())
    print(f"{col}: {100*(1 - failed/len(df)):.4f}% parsed, {failed} failed, "
          f"range {parsed.min()} to {parsed.max()}")

order_dt = pd.to_datetime(df["order date (DateOrders)"], errors="coerce")
ship_dt = pd.to_datetime(df["shipping date (DateOrders)"], errors="coerce")
print("Rows where shipping date < order date:", int((ship_dt < order_dt).sum()))

order date (DateOrders): 100.0000% parsed, 0 failed, range 2015-01-01 00:00:00 to 2018-01-31 23:38:00


shipping date (DateOrders): 100.0000% parsed, 0 failed, range 2015-01-03 00:00:00 to 2018-02-06 22:14:00


Rows where shipping date < order date: 0


## 11. Candidate target inspection (Step 7)

Two possible targets: `Late_delivery_risk` and `Delivery Status`. Both
represent the same underlying outcome.

In [13]:
print(df["Delivery Status"].value_counts())
print()
print(df["Late_delivery_risk"].value_counts())
print()
print(pd.crosstab(df["Delivery Status"], df["Late_delivery_risk"]))

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Late_delivery_risk      0      1
Delivery Status                 
Advance shipping    41592      0
Late delivery           0  98977
Shipping canceled    7754      0
Shipping on time    32196      0


## 12. Initial leakage notes

Testing whether `Days for shipping (real)` (only known post-shipment)
nearly determines the target.

In [14]:
derived_late = (df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]).astype(int)
match_fraction = (derived_late == df["Late_delivery_risk"]).mean()
print(f"Fraction where (real shipping days > scheduled) matches Late_delivery_risk: {match_fraction:.4f}")

print()
print("Shipping Mode vs Late_delivery_risk (row-normalized, i.e. % late per mode):")
print(pd.crosstab(df["Shipping Mode"], df["Late_delivery_risk"], normalize="index").round(3))

Fraction where (real shipping days > scheduled) matches Late_delivery_risk: 0.9755

Shipping Mode vs Late_delivery_risk (row-normalized, i.e. % late per mode):
Late_delivery_risk      0      1
Shipping Mode                   
First Class         0.047  0.953
Same Day            0.543  0.457
Second Class        0.234  0.766
Standard Class      0.619  0.381


`Days for shipping (real)` and `shipping date (DateOrders)` are flagged
as **post-outcome / high leakage risk** and should not be used as
predictive features for a delivery-lateness model. `Shipping Mode`, by
contrast, is set at order time and shows a real, informative
association with lateness — a legitimate candidate feature, not a
leakage source. Full discussion in
`artifacts/reports/dataco_schema_summary.md`.

## 13. Milestone 1 conclusions

- Dataset loads cleanly: 180,519 rows × 53 columns, 0 fully duplicate rows.
- `Product Description` is 100% missing; `Order Zipcode` is 86.24% missing.
- Three columns are exact duplicates of another column under a
  different name (`Order Customer Id`, `Product Category Id`,
  `Order Item Cardprod Id`).
- `Customer Email` and `Customer Password` are pre-masked constants,
  not real PII in this file as supplied. Several other customer/geo
  fields remain privacy-relevant and are flagged for later handling.
- `Late_delivery_risk` and `Delivery Status` are two encodings of the
  same outcome, only known post-delivery.
- `Days for shipping (real)` and `shipping date (DateOrders)` are
  flagged as leakage risks; `Days for shipment (scheduled)` and
  `Shipping Mode` are not.
- Recommended investigation for Milestone 2/3: binary late-delivery
  classification using `Late_delivery_risk` as the target, with
  post-outcome columns excluded from the feature set — to be
  re-verified with a full leakage audit before any training.

See `artifacts/reports/dataco_quality_report.md` and
`artifacts/reports/dataco_schema_summary.md` for the complete written
reports this notebook's analysis feeds into.